In [1]:
import pandas as pd
import glob, os
from tqdm.notebook import tqdm
import json
import plotly.express as px
import plotly.graph_objects as go
import geopandas as gpd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
CODIGO_COMUNIDAD = 28
ELECCIONES = 200403

# Datos electorales

In [17]:
def clean_column_names(df):
    df.columns = df.columns.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '').str.replace('-', '_')
    df.columns = df.columns.str.replace('.', '')
    df.columns = df.columns.str.replace("á", "a").str.replace("é", "e").str.replace("í", "i").str.replace("ó", "o").str.replace("ú", "u")
    return df

def map_party_names(df):
    mappings={'oldName1': 'newName1', 'oldName2': 'newName2'}
    df = df.rename(mappings)
    return df

In [18]:
data_path = "../../data/raw/resultados_electorales/congreso_por_municipio/"
df = None
for file in tqdm(glob.glob(data_path + "*.xlsx")):
    df_single = pd.read_excel(file, header=1, skiprows=4)
    year = os.path.splitext(file)[0].split("/")[-1][3:7]
    month = os.path.splitext(file)[0].split("/")[-1][7:9]
    df_single["year"] = int(year)
    df_single["month"] = int(month)
    df_single["yearmonth"] = year+month
    print("yearmonth: ", year+month)
    df_single = clean_column_names(df_single)
    cols = [c for c in df_single.columns if "unnamed" not in c]
    df_single = df_single[cols]
    df_single = map_party_names(df_single)
    #print(df_single.columns)
    if df is None:
        df = df_single
    else:
        df = pd.concat([df, df_single], ignore_index=True, sort=False)
df = df.fillna(0)

yearmonth:  199603
Index(['nombre_de_comunidad', 'codigo_de_provincia', 'nombre_de_provincia',
       'codigo_de_municipio', 'nombre_de_municipio', 'poblacion',
       'numero_de_mesas', 'total_censo_electoral', 'total_votantes',
       'votos_validos', 'votos_a_candidaturas', 'votos_en_blanco',
       'votos_nulos', 'pp', 'psoe', 'iu', 'ciu', 'eaj_pnv', 'cc', 'bng', 'hb',
       'erc', 'pa', 'ea', 'uv', 'lve', 'cha', 'uc', 'upv_bn', 'psm_ene', 'cdn',
       'lv_gv', 'prt', 'pcpe', 'ph', 'pas', 'upl', 'fea', 'icv_gorordo', 'lv',
       'cex', 'um', 'tc_pnc', 'priojano', 'pec', 'urcl', 'na', 'aun', 'cr',
       'sos', 'prepal', 'frepic', 'pspc', 'prcm', 'fpg', 'siex', 'prv', 'prim',
       'fei', 'nr', 'picc', 'ar', 'env', 'pae', 'pb', 'pcn', 'upra', 'uad',
       'adec', 'vdpa', 'n', 'alas', 'aba', 'prgu', 'lae', 'dsa', 'lg', 'piz',
       'pnm', 'por', 'afit', 'ccv', 'year', 'month', 'yearmonth'],
      dtype='object')
yearmonth:  201904
Index(['nombre_de_comunidad', 'codigo_de_provin

yearmonth:  200003
Index(['nombre_de_comunidad', 'codigo_de_provincia', 'nombre_de_provincia',
       'codigo_de_municipio', 'nombre_de_municipio', 'poblacion',
       'numero_de_mesas', 'total_censo_electoral', 'total_votantes',
       'votos_validos',
       ...
       'un', 'tpc', 'cives', 'mash', 'pdep', 'an', 'eh', 'year', 'month',
       'yearmonth'],
      dtype='object', length=113)
yearmonth:  201111
Index(['nombre_de_comunidad', 'codigo_de_provincia', 'nombre_de_provincia',
       'codigo_de_municipio', 'nombre_de_municipio', 'poblacion',
       'numero_de_mesas', 'total_censo_electoral', 'total_votantes',
       'votos_validos', 'votos_a_candidaturas', 'votos_en_blanco',
       'votos_nulos', 'pp', 'psoe', 'iu_lv', 'upyd', 'ciu', 'amaiur',
       'eaj_pnv', 'esquerra', 'equo', 'bng', 'cc_nc_pnc', 'compromis_q',
       'pacma', 'fac', 'eb', 'pa', 'pxc', 'prc', 'gbai', 'pum+j', 'pcpe',
       'anticapitalistas', 'piratacat', 'uce', 'ph', 'espana_2000', 'sain',
       'rps', 'h

In [19]:
df.head()

,nombre_de_comunidad,codigo_de_provincia,nombre_de_provincia,codigo_de_municipio,nombre_de_municipio,poblacion,numero_de_mesas,total_censo_electoral,total_votantes,votos_validos,votos_a_candidaturas,votos_en_blanco,votos_nulos,pp,psoe,iu,ciu,eaj_pnv,cc,bng,hb,erc,pa,ea,uv,lve,cha,uc,upv_bn,psm_ene,cdn,lv_gv,prt,pcpe,ph,pas,upl,fea,icv_gorordo,lv,cex,um,tc_pnc,priojano,pec,urcl,na,aun,cr,sos,prepal,frepic,pspc,prcm,fpg,siex,prv,prim,fei,nr,picc,ar,env,pae,pb,pcn,upra,uad,adec,vdpa,n,alas,aba,prgu,lae,dsa,lg,piz,pnm,por,afit,ccv,year,month,yearmonth,cs,podemos_iu_equo,vox,erc_sobiranistes,ecp_guanyem_el_canvi,jxcat_junts,eh_bildu,podemos_eu_mareas_en_comun_equo,cca_pnc,na+,compromis_2019,prc,pacma,front_republica,recortes_cero_gv,nca,pact,ara_mes_esquerra,gbai,pum+j,en_marea,pcte,el_pi,axsi,pcoe,avant_adelante_los_verdes,eb,cpm,somos_region,pcpa,uig_som_cuides,erpv,izqp,pcpc,ahora_canarias,cxg,ppso,cnv,cex_crex_prex,pr+,p_lib,cilu_linares,andecha_astur,"""jf""",pyln,fia,fe_de_las_jons,f8,solidaria,dpl,union_regionalista,centrados,vou,dp,pdsje_udec,izar,risa,c_21,+mas+,udt,ap_pdp,ucd,pce,cds,fn,pst,ee,psa_pa,pcc,b_psg,upc,ne,se,uce,pce_m_l,eu,eg,ac_cc,upv,cuc,ps,mfe,eupv,aeba,psm,lc_cosi,psa,env_urv,aegi,ce_fcc,fe_jons,fic,pnc,padl,cc1,idg,fe_i,lcr_mc,lcr,aci,pre,pnc1,pdb,mcate,asdc,ir,fa,em,pcu,pc,emk,bia,cir,partido_popular,partido_socialista_obrero_espanol,ciudadanos_partido_de_la_ciudadania,podemos,en_comu_podem,"unidad_popular:_izquierda_unida,_unidad_popular_en",compromis_podemos_es_el_moment,esquerra_republicana_de_catalunya_catalunya_si,democracia_i_llibertat_convergencia_democrates,euzko_alderdi_jeltzalea_partido_nacionalista_vasco,partido_animalista_contra_el_maltrato_animal,euskal_herria_bildu,union_progreso_y_democracia,coalicion_canaria___partido_nacionalista_canario,nos_candidatura_galega_bng_cg_foga_pcpg_pg,unio_democratica_de_catalunya,recortes_cero_grupo_verde,mes,partido_comunista_de_los_pueblos_de_espana,geroa_bai,el_pi___proposta_per_les_illes,ciudadanos_de_centro_democratico,escanos_en_blanco,falange_espanola_de_las_jons,x_la_izquierda_los_verdes,som_valencians,por_un_mundo_mas_justo,solidaridad_y_autogestion_internacionalista,los_verdes_ecopacifistas,partido_da_terra,"canarias_decide:_los_verdes,_unidad_del_pueblo_y_a",partido_libertario,partido_humanista,"ara,_pais_valencia",extremadura_unida_extremenos,partido_comunista_obrero_espanol,democracia_nacional,iniciativa_feminista,partido_regionalista_del_pais_leones,en_positiu,ciudadanos_libres_unidos,ciudadanos_rurales_agrupados,"libertad_navarra,_libertate_nafarra",avant_valencians,malaga_por_si,familia_y_vida,andaluces_de_jaen_unidos,independientes_por_aragon,foro_democrata,soluciona,"justizia_social,_participacion_ciudadana",muerte_al_sistema,partido_liberal_de_derechas,ongi_etorri,ruiz_mateos,lv_lv,pte_uc,par,aic,cg,psg_eg,av_mec,verde,acn,csd,png_pg,exu,pore,igc,ped,pam,grm,una,ua_cha,ub,ceu,arde,sev,mv,pancal,coa,pee_lv,r_x_c,tagoror,prb,pnem,ppr,lki_emk,pce_ml,cd,un,pnv,pte,psoe_h,ort,erfn,mc_oic,bnpg,c_upc,pg_pog_psg,bean,pcarl,pct,ufv,fjonsa,upn,oce_br,psar___psda,plib,urv,pnpv,unai,pdpcan,smim,psin,ule,e_catala,cic,p_cant,dscc,pprov,lc,ocebr_uce,cna,pippa,poc,cime,fe_uf,uf_fi_at,verdes,ea_eue,le,arm_ade,pap,upca,ua,gil,pgc,pln,e_u,ummp,pec_verde,nps,p_riojano,cng,ag,prex,sees,arcoiris,app,pvpa,mce,atf,aigranc,fpe,aa,fe_a,unida,udla,os,conceyu,piap,abe,pitrcg,if,p_tagoror,mces,us,ince,freixes,ccl,ap,psp_us,pdpc,fdc_edc,udc_idcc,ec_fed,fdi,asdci,aet,an18,rse,fut,caic,esb,pspv,indep,dcv,psg,fjons,afn,indep1,ura,lliga,uan,anepa_cp,fl,div,cja,cups,ccia,uab,uras,fni,ppcan,ms,asa,mfa,anv,indep2,indep3,indep4,dcar,udib,indep5,cup,pscan,faa,pim,indep6,prsv,icu,psde,indep7,indep8,pdg,indep9,indep10,indep11,aeicyu,cuir,cipye,pli,adc,bai,indep12,cai,psoe_progr,ic_v,bloc_verds,psm_en,uc_cds,lv_cm,fe,posi,ev_av,pil,es2000,pade,um_inme,ia,blm,fei_fe_2000,pcl,autonomo,ec,pkd,el_papi,a,prex_crex,dg,upz,cnc,peda,li_lit_ci,paviel,ir_pre,usi,pnca,inar,up_can,ural,ccs,cs

In [20]:
df.columns

Index(['nombre_de_comunidad', 'codigo_de_provincia', 'nombre_de_provincia',
       'codigo_de_municipio', 'nombre_de_municipio', 'poblacion',
       'numero_de_mesas', 'total_censo_electoral', 'total_votantes',
       'votos_validos',
       ...
       'ias', 'ucm', 'zu', 'fde', 'psda', 'amaga', 'pnt', 'ccse', 'pdn',
       'gphae'],
      dtype='object', length=671)

In [21]:
"""data_path = f"../../data/raw/resultados_electorales/congreso_por_municipio/02_{ELECCIONES}_1.xlsx"
df = pd.read_excel(data_path, header=1, skiprows=4)
year = os.path.splitext(data_path)[0].split("/")[-1][3:7]
month = os.path.splitext(data_path)[0].split("/")[-1][7:9]
df["Año"] = int(year)
df["Mes"] = int(month)
df["añomes"] = year+month
print("yearmonth: ", year+month)"""

'data_path = f"../../data/raw/resultados_electorales/congreso_por_municipio/02_{ELECCIONES}_1.xlsx"\ndf = pd.read_excel(data_path, header=1, skiprows=4)\nyear = os.path.splitext(data_path)[0].split("/")[-1][3:7]\nmonth = os.path.splitext(data_path)[0].split("/")[-1][7:9]\ndf["Año"] = int(year)\ndf["Mes"] = int(month)\ndf["añomes"] = year+month\nprint("yearmonth: ", year+month)'

## Filtrar por Madrid

In [22]:
df = df[df["Código de Provincia"] == CODIGO_COMUNIDAD]  # 28 = Madrid

KeyError: 'Código de Provincia'

In [23]:
df.shape

(121362, 671)

In [24]:
df.columns

Index(['nombre_de_comunidad', 'codigo_de_provincia', 'nombre_de_provincia',
       'codigo_de_municipio', 'nombre_de_municipio', 'poblacion',
       'numero_de_mesas', 'total_censo_electoral', 'total_votantes',
       'votos_validos',
       ...
       'ias', 'ucm', 'zu', 'fde', 'psda', 'amaga', 'pnt', 'ccse', 'pdn',
       'gphae'],
      dtype='object', length=671)

In [ ]:
# limpiar columnas index y rellenar NA
df = df.fillna(0)
# eliminar columnas con menso de x votos
#df_sum = pd.DataFrame(df.sum(axis = 0, skipna = True, numeric_only=True))
#df_sum[df_sum[0] == 0].index

In [ ]:
# crear codigo ine
df["CODIGOINE"] = df["Código de Provincia"].astype(int).astype(str).str.zfill(2) + df["Código de Municipio"].astype(int).astype(str).str.zfill(3)

In [ ]:
df.tail(5)

# Mapa Comunidad por munipios

In [ ]:
#map_path = "../../data/raw/mapa_municipios/Municipios_IGN-shp/Municipios_IGN.shp"
#geodf = gpd.read_file(map_path)
#geodf["COMUNIDAD"] = geodf["CODIGOINE"].apply(lambda x: str(x)[:2])
#geodf = geodf[geodf["COMUNIDAD"] == str(CODIGO_COMUNIDAD).zfill(2)]
#geodf_reduced = geodf[["CODIGOINE","geometry"]]
#geojson_save_path = "../../data/raw/mapa_municipios/Municipios_IGN-shp/madrid_municipios.geojson"
#guardar
#geodf_reduced.to_file(geojson_save_path, driver = "GeoJSON")

In [ ]:
geojson_path = "../../data/raw/mapa_municipios/Municipios_IGN-shp/madrid_municipios_reduced.geojson"

In [ ]:
#cargar
with open(geojson_path) as geofile:
    geojson = json.load(geofile)

# Mapa de ganadores

In [ ]:
key_columns = ['Nombre de Comunidad', 'Código de Provincia', 'Nombre de Provincia',
       'Código de Municipio', 'Nombre de Municipio', 'Población',
       'Número de mesas', 'Total censo electoral', 'Total votantes',
       'Votos válidos', 'Votos a candidaturas', 'Votos en blanco',
       'Votos nulos','Año', 'Mes', 'añomes', 'CODIGOINE']
party_columns = [x for x in df.columns if x not in key_columns]

In [ ]:
df["ganador"] = df[party_columns].idxmax(axis=1)

In [ ]:
df["ganador"].unique()

# Resultados electorales

In [ ]:
color_map = {'UCD':"#66AA00",
             'VOX':"#66AA00",
             'PSOE':"#EF553B", 
             'PODEMOS-IU':"#AB63FA",              
             'PP':"#636EFA", 
             'AP':"#636EFA",              
             'IU':"#2CA02C", 
             'CHA':"#B82E2E", 
             'UM':"#EECA3B", 
             'CC':"#17BECF", 
             'CIU':"#3366CC", 
             'ERC':"#FF9900", 
             'BNG':"#0DF9FF", 
             'HB':"#222A2A",
             'CDN':"#AF0038", 
             'EA':"#66AA00", 
             'EAJ-PNV':"#479B55", 
             'P.RIOJANO':"#86CE00"}

In [ ]:
# filtros
df_filtered = df[df["añomes"] == str(ELECCIONES)]
df_filtered = df_filtered[["ganador", "Nombre de Municipio", "CODIGOINE"]]

In [ ]:
df_filtered.shape

In [ ]:
fig = px.choropleth_mapbox(data_frame=df_filtered, geojson=geojson, locations='CODIGOINE', 
                           featureidkey="properties.CODIGOINE", color='ganador', mapbox_style="carto-positron", 
                           hover_data=["Nombre de Municipio", "ganador"], zoom=7, color_discrete_map=color_map,
                           center={"lat": 40.463667, "lon": -3.74922}, 
                           title= f"Elecciones Generales {str(ELECCIONES)[:4]}-{str(ELECCIONES)[-2:]} en Madrid",
                            )
fig.show()

# Evolución votos

In [ ]:
# evolucion partidos mas votados, lineas dejar rastro, filtro por municipio

In [ ]:
df.head()

In [ ]:

def get_town_series(df, cod_municipio, party_columns):
    df_tmp = df[df["Código de Municipio"] == cod_municipio]
    df_tmp = df_tmp[["Nombre de Municipio", "Población", "Votos en blanco", "Votos nulos", "Año","Mes"]+party_columns]
    print(df_tmp[party_columns].sum().sort_values(ascending=False)[:20])
    return df_tmp
   
df_municipio = get_town_series(df=df, cod_municipio=5, party_columns=party_columns)
df_municipio.head()

In [26]:
"""df2 = px.data.gapminder()
fig = px.scatter(df2, x="year", y="lifeExp", animation_frame="year", animation_group="country",
           color="continent", hover_name="continent",
           size_max=45, range_x=[1952,2007], range_y=[0,100])
fig.show()"""

'df2 = px.data.gapminder()\nfig = px.scatter(df2, x="year", y="lifeExp", animation_frame="year", animation_group="country",\n           color="continent", hover_name="continent",\n           size_max=45, range_x=[1952,2007], range_y=[0,100])\nfig.show()'

In [ ]:
"""df3 = px.data.gapminder()
fig = px.line(df3, x="year", y="lifeExp", color="continent", line_group="country", hover_name="country",
        line_shape="spline", render_mode="svg", animation_frame="year",range_x=[1952,2007], range_y=[0,100])
fig.show()"""